# Roomify — Colab Launcher

Run cells **top to bottom** on a fresh session.  After Cell 6 prints a `trycloudflare.com` URL, open it in your laptop browser.

| Cell | Purpose | One-time? |
|------|---------|-----------|
| 1 | Mount Google Drive, create folder layout | No (fast) |
| 2 | Clone repo + install deps | First run per Drive |
| 3 | Set HF_HOME to Drive-backed cache | No (fast) |
| 4 | Verify GPU | No (fast) |
| 5 | SD 1.5 smoke-test generation | First run (downloads weights) |
| 6 | Start Streamlit + Cloudflare tunnel | Every session |
| 7 | Reconnect helper (after disconnect) | On reconnect only |

In [ ]:
# ── Cell 1: Mount Google Drive and create folder structure ─────────────────
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for subdir in ['data', 'outputs', 'hf_cache']:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

# Symlink Drive folders into the repo working directory for convenience.
# (The repo is cloned to /content/roomify in Cell 2.)
print('Drive mounted.  Folder layout:')
for p in sorted(DRIVE_ROOT.iterdir()):
    print(' ', p)

In [ ]:
# ── Cell 2: Clone repo and install Python dependencies ─────────────────────
# Replace PLACEHOLDER_GITHUB_USER with your GitHub username.
REPO_URL = 'https://github.com/PLACEHOLDER_GITHUB_USER/roomify.git'
REPO_DIR = Path('/content/roomify')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists — pulling latest changes.')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# Symlink Drive data/outputs into the repo so paths.py resolves correctly.
import subprocess
DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for link_name in ['data', 'outputs']:
    link_path = REPO_DIR / link_name
    target = DRIVE_ROOT / link_name
    if not link_path.exists():
        link_path.symlink_to(target)
        print(f'Symlinked {link_path} -> {target}')

!pip install -q -r requirements.txt
print('\nInstallation complete.')

In [ ]:
# ── Cell 3: Point HF_HOME at the Drive-backed cache ────────────────────────
# Weights downloaded once persist across Colab sessions.
import os
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/roomify/hf_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')

print(f'HF_HOME={os.environ["HF_HOME"]}')
print(f'Cache size: ', end='')
!du -sh {HF_CACHE} 2>/dev/null || echo '(empty)'

In [ ]:
# ── Cell 4: Verify GPU and log to session-level file ───────────────────────
import subprocess, json, datetime
from pathlib import Path

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip() if result.returncode == 0 else 'GPU not found'
print('GPU:', gpu_info)

log = {
    'timestamp': datetime.datetime.utcnow().isoformat(),
    'gpu': gpu_info,
}
log_path = Path('/content/roomify/session_gpu.json')
log_path.write_text(json.dumps(log, indent=2))
print('Logged to', log_path)

In [ ]:
# ── Cell 5: SD 1.5 smoke-test (downloads weights on first run) ─────────────
# Weights (~4 GB) download to the Drive-backed HF cache set in Cell 3.
# Subsequent sessions skip the download and load from Drive (<30 s).
import sys, os
sys.path.insert(0, '/content/roomify/src')

import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path
from IPython.display import display

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()

generator = torch.Generator(device='cuda').manual_seed(42)
image = pipe(
    'A scandinavian bedroom, natural light, 4k interior design photography',
    negative_prompt='low quality, blurry, distorted proportions',
    num_inference_steps=20,
    generator=generator,
).images[0]

out_path = Path('/content/roomify/outputs/smoke_test.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
image.save(out_path)
print('Smoke test image saved to', out_path)
display(image)

In [ ]:
# ── Cell 6: Start Streamlit + Cloudflare quick tunnel ──────────────────────
# Phase 6 stub — full implementation wired in Phase 6.
import subprocess, time, re, threading
from pathlib import Path

REPO_DIR = Path('/content/roomify')
PORT = 8501

# -- Install cloudflared binary if missing --
CF_BIN = Path('/usr/local/bin/cloudflared')
if not CF_BIN.exists():
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF_BIN}
    !chmod +x {CF_BIN}
    print('cloudflared installed.')

# -- Launch Streamlit in background --
st_log = open('/content/streamlit.log', 'w')
st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'),
     '--server.port', str(PORT), '--server.headless', 'true'],
    cwd=str(REPO_DIR),
    stdout=st_log, stderr=st_log,
)
print(f'Streamlit PID {st_proc.pid} starting on port {PORT}...')
time.sleep(4)  # give Streamlit a moment to bind

# -- Launch Cloudflare tunnel --
cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    [str(CF_BIN), 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=cf_log, stderr=subprocess.PIPE,
    text=True,
)

# Parse the public URL from cloudflared stderr
tunnel_url = None
for _ in range(30):
    line = cf_proc.stderr.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'\n=== Roomify is live at: {tunnel_url} ===')
else:
    print('WARNING: Could not parse tunnel URL. Check /content/cloudflared.log')
    print('Fallback: use the pyngrok cell below.')

# -- pyngrok fallback (commented out; enable if cloudflared fails) --
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')
# tunnel = ngrok.connect(PORT)
# print(f'ngrok URL: {tunnel.public_url}')

In [ ]:
# ── Cell 7: Reconnect helper ────────────────────────────────────────────────
# Run this cell ONLY after a Colab disconnect (not on a fresh session).
# Re-mounts Drive and restarts the tunnel without redownloading weights.
import os, subprocess, time, re
from pathlib import Path
from google.colab import drive

print('Re-mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

# Restore HF_HOME
os.environ['HF_HOME'] = '/content/drive/MyDrive/roomify/hf_cache'

# Kill any stale Streamlit/cloudflared processes
!pkill -f streamlit 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# Re-run Cell 6 logic inline
REPO_DIR = Path('/content/roomify')
PORT = 8501
CF_BIN = Path('/usr/local/bin/cloudflared')

%cd {REPO_DIR}

st_log = open('/content/streamlit.log', 'w')
st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'),
     '--server.port', str(PORT), '--server.headless', 'true'],
    cwd=str(REPO_DIR),
    stdout=st_log, stderr=st_log,
)
time.sleep(4)

cf_proc = subprocess.Popen(
    [str(CF_BIN), 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True,
)
tunnel_url = None
for _ in range(30):
    line = cf_proc.stderr.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'\n=== Roomify reconnected at: {tunnel_url} ===')
else:
    print('WARNING: Could not parse tunnel URL. Check /content/cloudflared.log')